# Autoscaling configurations for publication

In [ ]:
# Basic imports
%matplotlib widget
from numpy import mean  
from ascal import AscalConfig, Ascal
from examples import aws_eu_west_1_c5m5r5
import csv
import random

In [ ]:
from math import sqrt

def sqrt_noise(x):
    delta = int(round(0.2*sqrt(x)))
    noise = random.randint(-delta, delta)
    return noise

def linear_noise(x):
    delta = int(round(0.1*x))
    noise = random.randint(-delta, delta)
    return noise

def generate_trace_file(file="trace.txt", duration=24*60, offset=0, ramp_up=0.25, plateau=0.5, 
                        ramp_down=0.25, start_value=1, end_value=10, noise=lambda x: 0):

    # Segment durations
    ramp_up = int(round(duration * ramp_up))
    plateau = int(round(duration * plateau))
    ramp_down = int(round(duration * ramp_down))
    valley = duration - (ramp_up + plateau + ramp_down)
    assert valley >= 0, "Duration too short for the given ramp/plateau proportions"

    def add_noise(value):
        return value + noise(value)

    values = []

    # Initial valley
    initial_valley = valley // 2
    for _ in range(initial_valley):
        noisy = add_noise(start_value)
        values.append(noisy)

    # Ramp up
    for i in range(ramp_up):
        base = start_value + (end_value - start_value) * i / (ramp_up - 1)
        noisy = add_noise(round(base))
        values.append(noisy)

    # Plateau
    for _ in range(plateau):
        noisy = add_noise(end_value)
        values.append(noisy)

    # Ramp down
    for i in range(ramp_down):
        base = end_value - (end_value - start_value) * i / (ramp_down - 1)
        noisy = add_noise(round(base))
        values.append(noisy)

    # Final valley
    for _ in range(valley - initial_valley):
        noisy = add_noise(start_value)
        values.append(noisy)

    # Save CSV
    values = values[-offset:] + values[:-offset]  # Apply offset
    with open(file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["value"])
        for v in values:
            writer.writerow([v])


In [ ]:
# Configuration files defining ASCAL problems
config_files = [
    "config_hpa_ca.yaml", 
    "config_fcma_reactive.yaml", 
    "config_fcma_predictive.yaml",
    "config_hpa_ca_fcma_reactive.yaml",
    "config_hpa_ca_fcma_predictive.yaml"
    ]
config_files_1h = [
    "config_hpa_ca_1h.yaml", 
    "config_fcma_reactive_1h.yaml", 
    "config_fcma_predictive_1h.yaml",
    "config_hpa_ca_fcma_reactive_1h.yaml",
    "config_hpa_ca_fcma_predictive_1h.yaml"
]
log_file = None

In [ ]:
# Generate the trace files
random.seed(0)
generate_trace_file(file="trapezoid1.csv", duration=24*60, offset=0, ramp_up=0.25, plateau=0.3, 
                    ramp_down=0.25, start_value=1, end_value=10, noise=sqrt_noise)
generate_trace_file(file="trapezoid2.csv", duration=24*60, offset=100, ramp_up=0.4, plateau=0.4, 
                    ramp_down=0.2, start_value=2, end_value=20, noise=sqrt_noise)
generate_trace_file(file="triangle1.csv", duration=1*60, offset=0, ramp_up=0.5, plateau=0, 
                    ramp_down=0.5, start_value=1, end_value=10, noise=sqrt_noise)
generate_trace_file(file="triangle2.csv", duration=1*60, offset=100, ramp_up=0.4, plateau=0.4, 
                    ramp_down=0.2, start_value=2, end_value=20, noise=sqrt_noise)

In [ ]:
# Simulate each configuration file
for config_file in config_files + config_files_1h:  # Change to config_files to run all configurations
    # Read the problem configuration file and validate it
    # Note that it is possible to validate any ASCAL configuration with method AscalConfig.validate_config()
    ascal_config = AscalConfig.get_from_config_yaml(config_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)

    # Create the autoscaling problem
    ascal_problem = Ascal(ascal_config, log=log_file)

    # Last time that can be simulated (last time in the trace)
    # Simulating time unit is seconds. Time starting from 0
    last_time = ascal_problem.last_time 
    print(f'Time range of the simulation: 0 - {last_time} seconds')

    # Run the autoscaling problem until the end. The argument of run() method is the last simulation time in seconds
    # Simulating time unit is seconds, so 3600 means 1 hour. Time starting from 0
    ascal_problem.run()

    # Get queue waiting times assuming each container is a server in an heterogenous D/D/m queue
    queue_waiting_times = ascal_problem.get_queue_waiting_times()
    avgs = {
        app_name: mean(waiting_times)
        for app_name, waiting_times in queue_waiting_times.items()
    }
    for app_name in dict(queue_waiting_times):
        queue_waiting_times[f"{app_name} Avg. QWT = {avgs[app_name]:.3f}"] = queue_waiting_times.pop(app_name)

    # Plot workloads
    if config_file == "config_hpa_ca.yaml" or config_file == "config_hpa_ca_1h.yaml":
        ascal_problem.plot(ascal_problem.get_workloads(), "Application Workloads", "req/s")

    # Plot autoscaling results
    print(f'-----------------------------------------------------------------')
    print(f'Plotting results for configuration file: {config_file}')
    print(f'-----------------------------------------------------------------')
    ascal_problem.plot(ascal_problem.get_performances(), "Application Performances", "req/s")
    cluster_cost = ascal_problem.get_cluster_cost()
    total_cost_str = f"Total cost = {sum(cluster_cost)/3600:.3f} $"
    ascal_problem.plot({total_cost_str: cluster_cost}, "Cluster Cost", "$/hour")
    ascal_problem.plot(queue_waiting_times, "Queue Waiting Times (L/μ)")